# UPanda Tutorial: DESI BAO DR2 Analysis

This tutorial demonstrates how to use UPanda to perform cosmological parameter inference using DESI BAO DR2 data with neural network emulators.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Loading Emulator Models](#loading-models)
3. [Configuring the MCMC Run](#configuration)
4. [Running MCMC Sampling](#running-mcmc)
5. [Visualizing Results with GetDist](#visualization)
6. [Understanding the Output](#understanding-output)


## 1. Setup and Installation {#setup}

First, let's set up UPanda and enable JAX for fast computations. JAX provides JIT compilation and automatic differentiation, making likelihood evaluations much faster.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import os

# Configure JAX for high precision (64-bit floating point)
from jax import config as jax_config
jax_config.update("jax_enable_x64", True)

# Enable JAX in UPanda
import UPanda.config as config
config.set_jax_enabled(True)

# Import UPanda modules
import UPanda.run_panda as run_panda
from UPanda.integrated_model_jax import IntegratedModel

print("✓ UPanda setup complete!")
print(f"✓ JAX enabled: {config.get_jax_enabled()}")


## 2. Loading Emulator Models {#loading-models}

UPanda uses neural network emulators to rapidly predict cosmological observables from parameter values. These emulators are trained to reproduce the output of expensive cosmological codes (like CLASS or CAMB) but run orders of magnitude faster.

For DESI BAO DR2, we need to load the pre-trained emulator model that predicts BAO observables (DM/rs, DH/rs, DV/rs) from cosmological parameters.

**Note**: The emulator models are stored in the `example_models/` directory. Adjust the path based on your directory structure.


In [ ]:
# Define the path to the emulator model
# The model consists of two files: bao_dr2_emu.pkl and bao_dr2_emu_model.h5
model_path = "../example_models/bao_dr2_emu"

# Initialize and load the emulator model
# IntegratedModel wraps the Keras/JAX model with preprocessing/postprocessing
bao_emu_model = IntegratedModel(None, None, None, None)
bao_emu_model.restore(model_path)

print("✓ Emulator model loaded successfully!")
print(f"✓ Model parameter names: {bao_emu_model.parameter_names}")


## 3. Configuring the MCMC Run {#configuration}

UPanda uses YAML configuration files to define the MCMC setup. These files specify:

- **Variable parameters**: Parameters to sample in the MCMC (with prior ranges)
- **Fixed parameters**: Parameters held constant during the run
- **Nuisance parameters**: Additional parameters specific to the likelihood

The configuration file for DESI BAO DR2 is located at `configs/config_example_desi_bao.yaml`. Let's set up the experiment configuration.


In [ ]:
# Set up the emulator model list
# For DESI BAO, we only need one emulator model
# The outer list corresponds to experiments, inner list to multiple models per experiment
total_emu_model_list = [[bao_emu_model]]

# Specify which experiments to use
# This tells UPanda which likelihood class to instantiate
experiments_list = ["desi_bao_dr2"]

# Compression vectors (empty for BAO - no compression needed)
compression_list = [[]]

# Path to configuration file
config_file = "../configs/config_example_desi_bao.yaml"

# Additional arguments for the likelihood
# For DESI BAO DR2, no special arguments are needed
additional_args = {}

print("✓ Configuration set up!")
print(f"  Experiments: {experiments_list}")
print(f"  Config file: {config_file}")


## 4. Running MCMC Sampling {#running-mcmc}

Now we'll run the MCMC sampler. Key parameters:

- **n_samples**: Total number of MCMC samples to generate
- **n_walkers**: Number of parallel walkers (chains). More walkers help explore the parameter space
- **sampler**: Which sampler to use. Options include:
  - `"emcee"`: Ensemble MCMC (default, good general purpose)
  - `"nautilus"`: Nested sampling
  - `"ultranest"`: UltraNest nested sampling  
  - `"hmc"`: Hamiltonian Monte Carlo (requires JAX)

**Note**: For production runs, use `n_samples >= 5e5` and `n_walkers >= 128` to ensure good convergence.


In [ ]:
# Run MCMC sampling
# For tutorial purposes, we use fewer samples - increase for production runs
print("Starting MCMC sampling...")
print("This may take several minutes depending on your hardware.")

params_chain, loglike = run_panda.run_mcmc(
    total_emu_model_list=total_emu_model_list,
    experiments_list=experiments_list,
    compression_list=compression_list,
    filename_config=config_file,
    n_samples=50000,  # Use 5e5 or more for production
    n_walkers=64,      # Use 128 or more for production
    sampler="emcee",  # Options: "emcee", "nautilus", "ultranest", "hmc"
    additional_args=additional_args
)

print("\n✓ MCMC sampling complete!")
print(f"  Chain shape: {params_chain.shape}")
print(f"  Parameter names: {params_chain.dtype.names}")
print(f"  Log-likelihood range: [{np.min(loglike):.2f}, {np.max(loglike):.2f}]")


## 5. Visualizing Results with GetDist {#visualization}

GetDist is a powerful tool for analyzing and visualizing MCMC chains. It provides:
- Corner plots (triangle plots) showing parameter correlations
- 1D marginalized distributions
- Summary statistics (means, credible intervals)
- Publication-quality plots

Let's convert our chain to GetDist format and create visualizations.


In [ ]:
# Install getdist if not already available
# !pip install getdist

# Convert chain to GetDist format
# GetDist expects a 2D array (n_samples, n_params) and parameter names
import getdist
from getdist import MCSamples, plots

# Extract parameter names and values from structured array
param_names = list(params_chain.dtype.names)
chain_array = np.column_stack([params_chain[name] for name in param_names])

# Create GetDist MCSamples object
# This object contains the chain and metadata
samples = MCSamples(
    samples=chain_array,
    names=param_names,
    labels=[name.replace('_', ' ').title() for name in param_names],
    label="DESI BAO DR2"
)

print("✓ GetDist samples created!")
print(f"  Number of samples: {len(samples.samples)}")
print(f"  Number of parameters: {samples.num_vars}")


In [ ]:
# Create corner plot (triangle plot)
# This shows 1D marginalized distributions on the diagonal
# and 2D contours showing parameter correlations
g = plots.get_subplot_plotter()
g.triangle_plot([samples], filled=True, contour_colors=['steelblue'])
plt.show()


In [ ]:
# Print summary statistics
# GetDist provides convenient methods for computing statistics
print("\n" + "="*60)
print("Parameter Constraints (Mean ± Std)")
print("="*60)
for name in param_names:
    mean = np.mean(params_chain[name])
    std = np.std(params_chain[name])
    # Get 68% credible interval
    lower = np.percentile(params_chain[name], 16)
    upper = np.percentile(params_chain[name], 84)
    print(f"{name:15s}: {mean:8.5f} ± {std:8.5f}  [{lower:8.5f}, {upper:8.5f}]")
print("="*60)


## 6. Understanding the Output {#understanding-output}

### Parameter Chain
The `params_chain` is a structured numpy array containing the MCMC samples. Each row is a sample, and each column corresponds to a cosmological parameter. This format makes it easy to access parameter values: `params_chain['omega_cdm']` gives all samples of omega_cdm.

### Log-Likelihood
The `loglike` array contains the log-likelihood values for each sample. This can be used to:
- Assess convergence (log-likelihood should stabilize)
- Identify outliers or problematic samples
- Compute Bayesian evidence (with nested samplers)

### Key Parameters for DESI BAO
- **omega_cdm**: Cold dark matter density (Ω_cdm)
- **omega_b**: Baryon density (Ω_b)
- **h**: Hubble constant (H₀/100, where H₀ is in km/s/Mpc)
- **n_s**: Scalar spectral index (primordial power spectrum tilt)
- **sigma8**: Amplitude of matter fluctuations on 8 Mpc/h scales

### Derived Parameters
You can compute derived parameters from the chain. For example, omega_m (total matter density) can be computed as:
$$\Omega_m = \frac{\Omega_{cdm} + \Omega_b + \Omega_\nu}{h^2}$$

### Next Steps
- Increase `n_samples` for better convergence
- Check autocorrelation time to ensure sufficient sampling
- Compare with other cosmological datasets
- Combine with other likelihoods (e.g., CMB data from Planck)


In [ ]:
# Example: Check convergence by plotting log-likelihood evolution
plt.figure(figsize=(12, 4))
plt.plot(loglike, alpha=0.5, linewidth=0.5)
plt.xlabel('Sample Number', fontsize=12)
plt.ylabel('Log-Likelihood', fontsize=12)
plt.title('Log-Likelihood Evolution During MCMC', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check if chain has converged (log-likelihood should be relatively stable)
mean_loglike = np.mean(loglike[-10000:])  # Last 10k samples
std_loglike = np.std(loglike[-10000:])
print(f"\nLog-likelihood statistics (last 10k samples):")
print(f"  Mean: {mean_loglike:.2f}")
print(f"  Std:  {std_loglike:.2f}")
if std_loglike < 5:
    print("  ✓ Chain appears converged (low variance in log-likelihood)")
else:
    print("  ⚠ Chain may need more samples (high variance in log-likelihood)")


In [ ]:
# Example: Compute derived parameter omega_m
# omega_m = (omega_cdm + omega_b + omega_nu) / h^2
# For simplicity, we assume omega_nu ≈ 0 (massless neutrinos)
omega_m = (params_chain['omega_cdm'] + params_chain['omega_b']) / (params_chain['h']**2)

# Compute statistics
mean_om = np.mean(omega_m)
std_om = np.std(omega_m)
lower_om = np.percentile(omega_m, 16)
upper_om = np.percentile(omega_m, 84)

print("Derived Parameter: Omega_m")
print(f"  Mean: {mean_om:.5f}")
print(f"  Std:  {std_om:.5f}")
print(f"  68% CI: [{lower_om:.5f}, {upper_om:.5f}]")

# Plot the distribution
plt.figure(figsize=(8, 5))
plt.hist(omega_m, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(mean_om, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_om:.4f}')
plt.axvline(lower_om, color='orange', linestyle=':', linewidth=2, label='68% CI')
plt.axvline(upper_om, color='orange', linestyle=':', linewidth=2)
plt.xlabel('Ω_m', fontsize=14)
plt.ylabel('Probability Density', fontsize=14)
plt.title('Posterior Distribution of Ω_m', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# UPanda Tutorial: DESI BAO DR2 Analysis

This tutorial demonstrates how to use UPanda to perform cosmological parameter inference using DESI BAO DR2 data with neural network emulators.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Loading Emulator Models](#loading-models)
3. [Configuring the MCMC Run](#configuration)
4. [Running MCMC Sampling](#running-mcmc)
5. [Visualizing Results with GetDist](#visualization)
6. [Understanding the Output](#understanding-output)


## 1. Setup and Installation {#setup}

First, let's set up UPanda and enable JAX for fast computations.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import os

# Configure JAX for high precision (64-bit)
from jax import config as jax_config
jax_config.update("jax_enable_x64", True)

# Enable JAX in UPanda
import UPanda.config as config
config.set_jax_enabled(True)

# Import UPanda modules
import UPanda.run_panda as run_panda
from UPanda.integrated_model_jax import IntegratedModel

print("UPanda setup complete!")
print(f"JAX enabled: {config.get_jax_enabled()}")


## 2. Loading Emulator Models {#loading-models}

UPanda uses neural network emulators to rapidly predict cosmological observables. 
For DESI BAO DR2, we need to load the pre-trained emulator model that predicts 
BAO observables (DM/rs, DH/rs, DV/rs) from cosmological parameters.

**Note**: The emulator models are stored in the `example_models/` directory.


In [ ]:
# Define the path to the emulator model
# Adjust this path based on your directory structure
model_path = "../example_models/bao_dr2_emu"

# Initialize and load the emulator model
bao_emu_model = IntegratedModel(None, None, None, None)
bao_emu_model.restore(model_path)

print("Emulator model loaded successfully!")
print(f"Model parameter names: {bao_emu_model.parameter_names}")


ak_system_packages alike look again at UFalcon th

In [ ]:
# Set up the emulator model list
# For DESI BAO, we only need one emulator model
total_emu_model_list = [[bao_emu_model]]

# Specify which experiments to use
experiments_list = ["desi_bao_dr2"]

# Compression vectors (empty for BAO)
compression_list = [[]]

# Path to configuration file
config_file = "../configs/config_example_desi_bao.yaml"

# Additional arguments for the likelihood
additional_args = {}

print("Configuration set up!")
print(f"Experiments: {experiments_list}")
print(f"Config file: {config_file}")


is is working!! 

In [ ]:
# Run MCMC sampling
# For tutorial purposes, we use fewer samples - increase for production runs
params_chain, loglike = run_panda.run_mcmc(
    total_emu_model_list=total_emu_model_list,
    experiments_list=experiments_list,
    compression_list=compression_list,
    filename_config=config_file,
    n_samples=50000,  # Use 5e5 or more for production
    n_walkers=64,      # Use 128 or more for production
    sampler="emcee",  # Options: "emcee", "nautilus", "ultranest", "hmc"
    additional_args=additional_args
)

print("MCMC sampling complete!")
print(f"Chain shape: {params_chain.shape}")
print(f"Parameter names: {params_chain.dtype.names}")


! Like look 

In [ ]:
# Convert chain to GetDist format
# GetDist expects a 2D array (n_samples, n_params) and parameter names
import getdist
from getdist import MCSamples, plots

# Extract parameter names and values
param_names = list(params_chain.dtype.names)
chain_array = np.column_stack([params_chain[name] for name in param_names])

# Create GetDist MCSamples object
samples = MCSamples(
    samples=chain_array,
    names=param_names,
    labels=[name.replace('_', ' ') for name in param_names],
    label="DESI BAO DR2"
)

print("GetDist samples created!")


In [ ]:
# Create corner plot
g = plots.get_subplot_plotter()
g.triangle_plot([samples], filled=True)
plt.show()
